# 01 — Match source CBGs to ZCTAs and build the panel

**What we are doing:** Dewey reports home Census Block Groups, but the Census and driving-distance tables use ZCTAs. We apply a crosswalk to allocate each CBG count across ZCTAs, then aggregate to origin × destination × month. **These crosswalks are provisional.**

Formula: $Y_{ijt}=\sum_b w_{bi}D_{bjt}$. Do not assume a missing reported origin means zero true visits.

In [ ]:
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
PRIVATE=ROOT/'data'/'private'; PROCESSED=ROOT/'data'/'processed';PROCESSED.mkdir(exist_ok=True)
with zipfile.ZipFile(PRIVATE/'ECON499_Dewey_2024_4Malls.zip') as archive:
    flows=pd.read_csv(archive.open('Origin_Destination_Flows.csv'),dtype={'Origin_CBG':str})
acs=pd.read_csv(PRIVATE/'ECON499_ACS_2024_25_ZCTA_Checked.csv',dtype={'ZCTA':str})
acs=acs.rename(columns={'B19013_001E':'income','B19083_001E':'gini','B01003_001E':'population','B03002_004E':'black_nonhisp','B03002_003E':'white_nonhisp','B03002_012E':'hispanic'})
for col in ['income','gini','population','black_nonhisp','white_nonhisp','hispanic']:
    acs[col]=pd.to_numeric(acs[col],errors='coerce')
for col in ['income','gini','population']:
    acs.loc[acs[col]<0,col]=np.nan
acs['black_share']=acs.black_nonhisp/acs.population
acs['hispanic_share']=acs.hispanic/acs.population
distance=pd.read_excel(PRIVATE/'ECON499_100_Driving_Distances_RETESTED.xlsx',sheet_name='100 Routes',dtype={'Origin ZCTA':str})
distance=distance.rename(columns={'Origin ZCTA':'ZCTA','Destination Mall':'Mall','Driving Miles':'miles'})[['ZCTA','Mall','miles']]
assert len(distance)==100 and not distance.duplicated(['ZCTA','Mall']).any()

## Allocate counts
**What we are doing:** Multiply each CBG origin count by its ZCTA allocation weight. This is **not** a direct device count from the ZCTA. Check how much of the source data maps into our 25-ZCTA study area.

In [ ]:
for scenario,filename in [('2020_geometry_provisional','crosswalk_2020_geometry_provisional.csv'),('2010_equal_block_diagnostic','crosswalk_2010_equal_block_DIAGNOSTIC.csv')]:
    cw=pd.read_csv(PRIVATE/filename,dtype={'Origin_CBG':str,'ZCTA':str})
    assert (cw.weight>=0).all()
    assert cw.groupby('Origin_CBG').weight.sum().max()<=1.001
    mapped=flows.merge(cw,on='Origin_CBG',how='inner')
    mapped['allocated_flow']=mapped.Origin_Device_Count*mapped.weight
    coverage=mapped.allocated_flow.sum()/flows.Origin_Device_Count.sum()
    print(scenario,'share allocated to 25 study ZCTAs:',round(coverage,4))
    aggregated=mapped.groupby(['ZCTA','Mall','Month'],as_index=False).allocated_flow.sum().rename(columns={'allocated_flow':'flow'})
    full=pd.MultiIndex.from_product([sorted(acs.ZCTA),sorted(flows.Mall.unique()),range(1,13)],names=['ZCTA','Mall','Month']).to_frame(index=False)
    panel=full.merge(aggregated,on=['ZCTA','Mall','Month'],how='left')
    panel['unreported_origin_cell']=panel.flow.isna()
    # Diagnostic choice only: a missing origin is not necessarily a true zero.
    panel['flow']=panel.flow.fillna(0)
    panel=panel.merge(acs,on='ZCTA',how='left').merge(distance,on=['ZCTA','Mall'],how='left')
    panel['log_distance']=np.log1p(panel.miles)
    panel['log_income']=np.log(panel.income)
    panel['log_population']=np.log(panel.population)
    panel['Buckhead']=panel.Mall.isin(['Lenox Square','Phipps Plaza']).astype(int)
    assert len(panel)==1200
    panel.to_csv(PROCESSED/f'panel_{scenario}.csv',index=False)
    print('Rows:',len(panel),'unreported cells:',panel.unreported_origin_cell.sum())